## Imports

In [1]:
#anaconda prompt > conda install -c anaconda openpyxl

import openpyxl

from bs4 import BeautifulSoup
import requests
import os
from datetime import datetime
import time
import pandas as pd


import warnings
warnings.filterwarnings('ignore')

## Funcion para extraccion de datos Zooplus

In [2]:
# Conectar y obtener datos desde Zooplus

#Se obtiene el User Agent desde http://httpbin.org/get
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36", "Accept-Encoding":"gzip, deflate", "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8", "DNT":"1","Connection":"close", "Upgrade-Insecure-Requests":"1"}

#Funcion para obtener el precio de un producto
def get_zooplus_price(url):

    #Se obtiene el HTML del producto
    page = requests.get(url, headers=headers, verify=False)
    soup1 = BeautifulSoup(page.content, "html.parser")

    #Se extrae el precio del HTML
    try:
        price = soup1.find("span", { "class" : "z-price__amount z-price__amount--standard" }).text.strip()
       
    except:
        try:
            price = soup1.find("span", {"class" : "z-price__amount"}).text.strip() 
    
        except AttributeError:
            price = "No se ha podido obtener el precio para este producto."
        
    return price

#funciones para hora y fecha

def actual_date():
    now = datetime.now()
    date_now = now.date()
   
    return date_now

def actual_hour():
    now = datetime.now()
    hour_now = now.time()

    return hour_now



def buscar_maximo(primer_dato, segundo_dato):
    # Cargar los datos de la tabla en un DataFrame
    datos = pd.read_excel('ruta_del_archivo.xlsx', sheet_name='nombre_de_la_hoja', engine='openpyxl')
    
    # Filtrar los datos por el primer dato especificado y obtener el valor máximo de la segunda columna (SegundaDato)
    valor_a_comparar = datos[datos['LA'] == primer_dato]['SegundaDato'].max()
    
    # Filtrar los datos por los dos datos especificados y obtener el valor máximo de la tercera columna (Precio)
    valor_maximo = datos[(datos['LA'] == primer_dato) & (datos['SegundaDato'] == segundo_dato)]['Precio'].max()
    
    # Devolver el valor máximo encontrado
    return valor_maximo


## Configuracion del Excel

In [3]:
#Incluir la localizacion del fichero excel con los datos de Zooplus:
directory = os.getcwd()
path = directory + "\\Precios.xlsx"
#print(directory)

# Se abre el workbook y la hoja seleccionados (donde se guardo el documento por ultima vez)
wb_obj = openpyxl.load_workbook(path)
sheet_obj = wb_obj.active

# Obtener el numero de filas y columnas usadas
row_limit = sheet_obj.max_row
column_limit = sheet_obj.max_column
  
print("Total de filas:", row_limit)
print("Total de columnas:", column_limit)

#Introducir el numero de columna en la que se encuentra el EAN (A=1, B=2, C=3...)
ean_col = 3

#Introducir el numero de la fila en la que se encuentra el primer valor de EAN
first_ean_row = 2

#Introducir el numero de columna en la que se encuentra el precio
precio_col = 4


Total de filas: 4025
Total de columnas: 4


# ***EJECUTAR LA TAREA DE AUTORRELLENADO***

In [4]:
# Cerrar el fichero Excel antes de ejecutar esta celda

#Introducir el formato de la web que se va a buscar
web_prefix = "https://www.zooplus.es/search/results?q="

date = actual_date()
hour = actual_hour()
time1 = time.time()

print(("INICIADO EL {} A LAS {} HORAS").format(date, hour))

for i in range(first_ean_row, row_limit + 1): 
    cell_obj = sheet_obj.cell(row = i, column = ean_col)
    ean = str(cell_obj.value)
    
    if ean == None:
        ean = ""
    
    ean = str(ean).zfill(12)
    url = web_prefix+ean
    try:
        price = get_zooplus_price(url)
        print(("{}: {} tiene un precio de: {}").format(i, ean, price))
    
        #Se introduce el precio en el Excel
        c1 = sheet_obj.cell(row = i, column = precio_col)
        c1.value = price
    except:
        print("Hemos tenido un error de conexión, voy a descansar 10 segundos. ZZZZZZzzzzz...")
        time.sleep(10)
        print("Buena siesta!!! Vuelvo a la carga!!!!")
        try:
            price = get_zooplus_price(url)
            print(("{}: {} tiene un precio de: {}").format(i, ean, price))
    
            #Se introduce el precio en el Excel
            c1 = sheet_obj.cell(row = i, column = precio_col)
            c1.value = price
        except:
            break
    
#Se guarda el fichero Excel con los cambios aplicados
wb_obj.save(path)
time2 = time.time()
hour1 = actual_hour()
dif_time = time2 - time1
print(("FINALIZADO EL {} A LAS {} HORAS").format(date, hour1))
print(("REALIZADO EN: {} minutos").format(dif_time/60))
print("ARCHIVO GUARDADO / PROGRAMA FINALIZADO")

INICIADO EL 2024-01-03 A LAS 10:30:33.139128 HORAS
2: 4062911008618 tiene un precio de: 21,99 €
3: 4062911004481 tiene un precio de: 13,99 €
4: 4260077045496 tiene un precio de: 16,99 €
5: 052742047447 tiene un precio de: 35,99 €
6: 5000161033072 tiene un precio de: 3,29 €
7: 052742025209 tiene un precio de: 13,79 €
8: 4260358514796 tiene un precio de: 4,79 €
9: 4260358513584 tiene un precio de: 1,99 €
10: 4062911021891 tiene un precio de: 18,99 €
11: 052742211107 tiene un precio de: 13,99 €
12: 4054651005422 tiene un precio de: 2,79 €
13: 4021158716908 tiene un precio de: 14,99 €
14: 4004218726338 tiene un precio de: 12,99 €
15: 8437013576192 tiene un precio de: 2,69 €
16: 3065890138254 tiene un precio de: 10,29 €
17: 4062911000636 tiene un precio de: 3,49 €
18: 4260358517513 tiene un precio de: 2,79 €
19: 4062911020191 tiene un precio de: 1,29 €
20: 022517443064 tiene un precio de: 5,79 €
21: 4062911003644 tiene un precio de: 6,49 €
22: 4054651012222 tiene un precio de: No se ha podi